<a href="https://colab.research.google.com/github/bkhn85dn/xac_dinh_va_phan_xu_diem_sai/blob/main/reduced_pkl_then_improve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
!rm -rf pose_jsons

Cấu hình biến môi trường in ra nhiều thông tin hay không?

In [55]:
DETAIL_PRINT = False

In [56]:
!pip install smplx chumpy trimesh

In [57]:
import joblib
import json
import os
import copy
import sys
import numpy as np
import torch
import smplx
from pathlib import Path
import pdb
import inspect  # 1. Thêm import inspect ở đây
import shutil
from datetime import datetime
import gdown
from zoneinfo import ZoneInfo
import cv2
import os
# Lấy thời gian hiện tại với múi giờ Hồ Chí Minh
now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
# Định dạng thành xâu YYMMDD_hhmm
AT_TIME = now.strftime("%y%m%d_%H%M")

tải tệp cần thiết để xử Iý cấu hình

In [58]:
def download_files():
    """Tải tệp từ Google Drive nếu chưa tồn tại cục bộ"""
    for filename, file_id in GOOGLE_DRIVE_IDS.items():
        if not os.path.exists(filename):
            if not DETAIL_PRINT:
              print(f"Đang tải {filename} từ Google Drive...")
            url = f'https://drive.google.com/uc?id={file_id}'
            gdown.download(url, filename, quiet=False)
        else:
            if not DETAIL_PRINT:
              print(f"Tệp {filename} đã tồn tại, bỏ qua tải xuống.")
#Tải file smpl
GOOGLE_DRIVE_IDS = {
    "smpl.zip" : "1axzI_DohdbZfhJfsc3gWDHQ8OkVVcVJ4",
    "calib_from_cam.zip" : "16m1RVsMvzEdrI5uQ-mxF9_sJcqsFfeOz"
}
download_files()

Tệp smpl.zip đã tồn tại, bỏ qua tải xuống.
Tệp calib_from_cam.zip đã tồn tại, bỏ qua tải xuống.


In [59]:
!unzip smpl.zip

Archive:  smpl.zip
replace smpl/basicModel_f_lbs_10_207_0_v1.0.0.pkl? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [60]:
# --- CẤU HÌNH ĐƯỜNG DẪN GOOGLE DRIVE ---
GOOGLE_DRIVE_IDS = {
    "Axel_1_cam_1.pkl" : "1y53im5EvAs4pIH5N-KOaKHNFBd6Ku10e",
    "Axel_1_cam_2.pkl" : "1AT2Wm-chzwcI5AvzM7jqrjzL7qC5UvZL",
    "Axel_1_cam_2.mp4" : "1AHAmdbmia_Fv4AVTj6bBGg2THOcgqBTB",
    "Axel_1_cam_1.mp4" : "1yJTVqEs79hcFyF3-Q1F5rFb8FD1azqM-",
    "Axel_1_cam_1_h36m.npy" :	"1_IC_tMzEoYwpmYv4_0AOWZvukFKcdM89",
    "Axel_1_cam_2_h36m.npy" : "10BRYpLB1Eh4jI9ZW7NqIUFuLNN5dunJU",
    "Axel_1_cam_3.pkl" : "1z9oXr6NPlqleZBwMswU-AuZzsuvyi0dm",
    "Axel_1_cam_3_h36m.npy" : "1TdqIMWQ8JjK0Acf_CJjndXG0peMbrvWP",
    "Axel_1_cam_3.mp4" :  "1sjz6Rd-hR7ls7UdEtP7T_-PID6uVKV0H",
    "Axel_1_cam_1.json" :	"1hVQHVDQO5H78XowmKeAiceQc1UVNWqdY",
    "Axel_1_cam_2.json" : "1VTh265g7yKwKO7UNO0m4A0K16xTuXsqk",
    "Axel_1_cam_3.json" : "1ghApUnWU8IgwqRMteeHiTu_IL2xZjL6n",
    # Hãy thêm SMPL_NEUTRAL.pkl vào dict nếu bạn có ID trên Drive, ví dụ:
    "SMPL_NEUTRAL.pkl" : "1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll"
}


In [61]:
download_files()

Tệp Axel_1_cam_1.pkl đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_2.pkl đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_2.mp4 đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_1.mp4 đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_1_h36m.npy đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_2_h36m.npy đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_3.pkl đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_3_h36m.npy đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_3.mp4 đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_1.json đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_2.json đã tồn tại, bỏ qua tải xuống.
Tệp Axel_1_cam_3.json đã tồn tại, bỏ qua tải xuống.
Tệp SMPL_NEUTRAL.pkl đã tồn tại, bỏ qua tải xuống.


Tính toán fps từ tệp pkl, nếu không thấy thì mới đọc tệp mp4

Cài đặt môi trường trước:

In [62]:
!pip install opencv-python

In [63]:
def get_fps_from_video(video_path, default_fps=30):
    """Đọc FPS gốc từ tệp video MP4."""
    if not os.path.exists(video_path):
        return default_fps

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return default_fps

    # Lấy thuộc tính FPS từ video
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    # Nếu đọc được FPS hợp lệ (>0), làm tròn thành số nguyên
    if fps > 0:
        return int(round(fps))
    return default_fps

def check_fps_in_pkl(person_data):
    # Thử tìm các key phổ biến thường được dùng để lưu fps
    if 'fps' in person_data:
        return int(person_data['fps'])
    elif 'frame_rate' in person_data:
        return int(person_data['frame_rate'])

    # Nếu muốn xem tệp pkl đang chứa những thông tin gì, bạn có thể print:
    # print("Các trường dữ liệu trong PKL:", person_data.keys())
    return None

def auto_detect_fps(pkl_path):
    """Hàm tự động tìm FPS tốt nhất có thể."""
    # 1. Thử đọc từ tệp PKL
    try:
        data = load_pkl_data(pkl_path)
        fps_from_pkl = check_fps_in_pkl(data)
        if fps_from_pkl is not None:
            print(f"   -> Đã tìm thấy thông số FPS trong tệp PKL: {fps_from_pkl}")
            return fps_from_pkl
    except:
        pass

    # 2. Thử tìm tệp MP4 tương ứng (ví dụ: Axel_1_cam_1.pkl -> Axel_1_cam_1.mp4)
    video_path = pkl_path.replace('.pkl', '.mp4')
    fps_from_video = get_fps_from_video(video_path, default_fps=None)

    if fps_from_video is not None:
        print(f"   -> Đã đọc được FPS từ video {video_path}: {fps_from_video}")
        return fps_from_video

    # 3. Mặc định
    print("   -> Không tìm thấy FPS từ PKL hay MP4, sử dụng mặc định: 30")
    return 30

Import thêm các thư viện cần thiết

In [64]:
import os
import copy
import json
import joblib
import inspect
import smplx
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from datetime import datetime
from pathlib import Path
from scipy.interpolate import CubicSpline
# --- THÊM THƯ VIỆN ĐỂ XỬ LÝ GÓC XOAY (SLERP) ---
from scipy.spatial.transform import Rotation as R
from scipy.spatial.transform import Slerp
# -----------------------------------------------



# Bản vá lỗi cho NumPy 2.0+ (Giữ nguyên từ code của bạn)
if not hasattr(inspect, "getargspec"):
    inspect.getargspec = inspect.getfullargspec

for name in ['int', 'float', 'bool', 'complex', 'object', 'str', 'unicode']:
    if not hasattr(np, name):
        if name == 'unicode':
            setattr(np, name, str)
        else:
            setattr(np, name, eval(name))


Các hằng số toàn cục

In [65]:
# Cấu hình cơ bản
ORIGINAL_FPS = 60  # Giả định video gốc đang ở 30 FPS
TARGET_FPS = 30    # FPS mục tiêu muốn giảm xuống
COLOR_RIGHT = 'red'
COLOR_LEFT = 'blue'
COLOR_TRUNK = 'gray'

# --- ĐỊNH NGHĨA KHUNG XƯƠNG ĐỂ VẼ ANIMATION ---
SMPL_JOINT_MAP = {
    # Thân
    "pelvis": 0, "neck": 12, "spine1": 3, "spine2": 6, "spine3": 9,
    # Chân
    "right_hip": 2, "right_knee": 5, "right_ankle": 8, "right_foot": 11,
    "left_hip": 1, "left_knee": 4, "left_ankle": 7, "left_foot": 10,
    # Tay
    "right_shoulder": 17, "right_elbow": 19, "right_hand": 21,
    "left_shoulder": 16, "left_elbow": 18, "left_hand": 20
}

# Bones SMPL phân chia chi tiết
# SMPL BONES (Chia Trái, Phải, Thân để tô màu)
BONES_SMPL = {
    "trunk": [
        ("pelvis", "spine1"), ("spine1", "spine2"), ("spine2", "spine3"), ("spine3", "neck")
    ],
    "right": [ # Tay phải, Chân phải (Màu Đỏ)
        ("pelvis", "right_hip"), ("right_hip", "right_knee"), ("right_knee", "right_ankle"), ("right_ankle", "right_foot"),
        ("neck", "right_shoulder"), ("right_shoulder", "right_elbow"), ("right_elbow", "right_hand")
    ],
    "left": [  # Tay trái, Chân trái (Màu Xanh)
        ("pelvis", "left_hip"), ("left_hip", "left_knee"), ("left_knee", "left_ankle"), ("left_ankle", "left_foot"),
        ("neck", "left_shoulder"), ("left_shoulder", "left_elbow"), ("left_elbow", "left_hand")
    ]
}

# Bộ xương H36M (Dữ liệu NPY) - Đã chia màu
BONES_H36M = {
    "trunk": [(0, 7), (7, 8), (8, 9), (9, 10)],
    "right": [(0, 1), (1, 2), (2, 3), (8, 14), (14, 15), (15, 16)], # Đỏ
    "left":  [(0, 4), (4, 5), (5, 6), (8, 11), (11, 12), (12, 13)]  # Xanh
}

# Các cặp khớp nối để vẽ khung xương dạng gậy (Stick figure)
BONES = [
    ("neck", "right_shoulder"), ("right_shoulder", "right_elbow"), ("right_elbow", "right_hand"),
    ("neck", "left_shoulder"), ("left_shoulder", "left_elbow"), ("left_elbow", "left_hand"),
    ("neck", "pelvis"),
    ("pelvis", "right_hip"), ("right_hip", "right_knee"), ("right_knee", "right_ankle"), ("right_ankle", "right_foot"),
    ("pelvis", "left_hip"), ("left_hip", "left_knee"), ("left_knee", "left_ankle"), ("left_ankle", "left_foot")
]

# --- CẤU HÌNH BỘ XƯƠNG CHO H36M (17 KHỚP TỪ NPY) ---
H36M_BONES = [
    (0, 1), (1, 2), (2, 3),                  # Chân phải
    (0, 4), (4, 5), (5, 6),                  # Chân trái
    (0, 7), (7, 8), (8, 9), (9, 10),         # Thân và đầu
    (8, 11), (11, 12), (12, 13),             # Tay trái
    (8, 14), (14, 15), (15, 16)              # Tay phải
]

# Ánh xạ các khớp tương đồng giữa Ground Truth (H36M) và SMPL để tính MPJPE
H36M_TO_SMPL_MAP = {
    0: 0,   # Pelvis (Gốc)
    1: 2,   # Right Hip
    2: 5,   # Right Knee
    3: 8,   # Right Ankle
    4: 1,   # Left Hip
    5: 4,   # Left Knee
    6: 7,   # Left Ankle
    8: 12,  # Neck
    11: 16, # Left Shoulder
    12: 18, # Left Elbow
    13: 20, # Left Wrist
    14: 17, # Right Shoulder
    15: 19, # Right Elbow
    16: 21  # Right Wrist
}

# Tên gọi của các khớp H36M để in báo cáo cho đẹp
JOINT_NAMES = {
    0: "Pelvis (Gốc)  ", 1: "Right Hip     ", 2: "Right Knee    ", 3: "Right Ankle   ",
    4: "Left Hip      ", 5: "Left Knee     ", 6: "Left Ankle    ",
    8: "Neck/Spine    ",
    11: "Left Shoulder ", 12: "Left Elbow    ", 13: "Left Wrist    ",
    14: "Right Shoulder", 15: "Right Elbow   ", 16: "Right Wrist   "
}

Hàm đọc mp4 để xuất ra các frames:

In [66]:
# --- HÀM BỔ SUNG: ĐỌC VIDEO THEO FRAME RANGE ---
def load_video_frames(video_path, start_frame, end_frame):
    """Đọc các khung hình từ start_frame đến end_frame của video mp4."""
    cap = cv2.VideoCapture(video_path)
    frames = []

    # Nhảy đến khung hình bắt đầu
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    for i in range(start_frame, end_frame + 1):
        ret, frame = cap.read()
        if not ret:
            break
        # OpenCV dùng BGR, Matplotlib dùng RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)

    cap.release()
    return frames

Hàm tìm Delta T tối ưu (Temporal Alignment): nó sẽ quét toàn bộ các cặp frame để tìm ra độ lệch xuất hiện nhiều nhất.

In [87]:
from scipy import stats

def find_optimal_time_offset(gt_seq, pred_seq):
    """
    Tìm Delta T tối ưu dựa trên phương pháp đối chiếu tư thế.
    gt_seq: Shape (N, 17, 3) - Ground Truth H36M
    pred_seq: Shape (M, 45+, 3) - Predict SMPL
    """
    n_gt = len(gt_seq)
    n_pred = len(pred_seq)
    offsets = []

    # Tách danh sách index dùng chung để slice (cắt) mảng
    gt_indices = list(H36M_TO_SMPL_MAP.keys())
    smpl_indices = list(H36M_TO_SMPL_MAP.values())

    # 1. Với mỗi frame trong bản dự đoán, tìm frame 'khớp nhất' trong bản GT
    for i in range(n_pred):
        # Lấy 14 khớp tương đồng của bản Predict (SMPL)
        p_frame = pred_seq[i][smpl_indices]

        # Dời về gốc Pelvis (Khớp 0) để loại bỏ sai lệch về vị trí đứng
        p_frame_aligned = p_frame - p_frame[0]

        # Xoay trục SMPL về Z-up để khớp với NPY (Cực kỳ quan trọng để đo đúng)
        p_frame_rot = np.stack([
            p_frame_aligned[:, 0],
            p_frame_aligned[:, 2],
            -p_frame_aligned[:, 1]
        ], axis=1)

        distances = []
        for j in range(n_gt):
            # Lấy 14 khớp tương đồng của bản GT (H36M)
            g_frame = gt_seq[j][gt_indices]

            # Dời về gốc Pelvis
            g_frame_aligned = g_frame - g_frame[0]

            # Tính khoảng cách giữa 14 khớp (Lúc này 2 mảng đều có shape là 14x3)
            dist = np.mean(np.linalg.norm(p_frame_rot - g_frame_aligned, axis=1))
            distances.append(dist)

        # Tìm frame GT có khoảng cách (sai số) nhỏ nhất với frame i
        best_gt_idx = np.argmin(distances)
        offsets.append(best_gt_idx - i) # Delta = index_GT - index_Pred

    # 2. Tìm giá trị Delta T xuất hiện thường xuyên nhất (Mode)
    delta_t = int(stats.mode(offsets, keepdims=True).mode[0])

    print(f"--- Temporal Alignment Analysis ---")
    print(f"Thống kê Delta T tìm thấy: {delta_t} frames")
    return delta_t

Nạp ma trận Regressor và định nghĩa Mapping

In [68]:
import numpy as np

# Tải ma trận Regressor (Kích thước: 25 x 6890)
# Nhớ đổi đường dẫn cho khớp với vị trí file của bạn
REGRESSOR_BODY25 = np.load('smpl/J_regressor_body25.npy')

# Định nghĩa 14 khớp tương đồng để tính MPJPE chéo
# Thứ tự: R_Ankle, R_Knee, R_Hip, L_Hip, L_Knee, L_Ankle, R_Wrist, R_Elbow, R_Shoulder, L_Shoulder, L_Elbow, L_Wrist, Neck, Nose
BODY25_14_IDX = [11, 10, 9, 12, 13, 14, 4, 3, 2, 5, 6, 7, 1, 0]
H36M_14_IDX = [3, 2, 1, 4, 5, 6, 16, 15, 14, 11, 12, 13, 8, 9]

# Trong mảng 14 khớp mới này, R_Hip ở index 2, L_Hip ở index 3

Hàm tính MPJPE chéo (Cross-Dataset MPJPE)

In [69]:
def compute_cross_mpjpe(verts, h36m_joints):
    """
    verts: Mảng tọa độ đỉnh SMPL của 1 frame (6890, 3)
    h36m_joints: Mảng tọa độ khớp H36M của 1 frame tương ứng (17, 3)
    Trả về MPJPE (khoảng cách lỗi trung bình trên 14 khớp).
    """
    # Bước 1: Nhân ma trận (25, 6890) @ (6890, 3) -> (25, 3) tọa độ Body25
    body25_joints = REGRESSOR_BODY25 @ verts

    # Bước 2: Lọc ra 14 khớp dùng chung
    pred_14 = body25_joints[BODY25_14_IDX]
    gt_14 = h36m_joints[H36M_14_IDX]

    # Bước 3: Root Alignment (Tính theo trung điểm 2 hông để công bằng cho cả 2 hệ)
    # L_Hip và R_Hip nằm ở index 2 và 3 trong mảng 14 khớp vừa tạo
    pred_root = (pred_14[2] + pred_14[3]) / 2.0
    gt_root = (gt_14[2] + gt_14[3]) / 2.0

    pred_14_aligned = pred_14 - pred_root
    gt_14_aligned = gt_14 - gt_root

    # Bước 4: Tính Euclidean distance và trung bình
    distances = np.linalg.norm(pred_14_aligned - gt_14_aligned, axis=-1)

    # Giả định dữ liệu đang ở đơn vị mét, nhân 1000 để hiển thị milimet
    return np.mean(distances) * 1000.0

Hàm tính Root-Aligned MPJPE:

In [70]:
def calculate_root_aligned_mpjpe(gt_frame, smpl_frame):
    """
    Tính MPJPE (Root-aligned) cho 1 khung hình.
    Trả về sai số trung bình của khung hình đó tính bằng milimet (mm).
    Giả định dữ liệu đầu vào đang ở đơn vị mét (m).
    """
    # Lấy tọa độ Pelvis làm gốc (Root)
    gt_root = gt_frame[0]
    smpl_root = smpl_frame[0]

    # Dời tọa độ về gốc 0,0,0
    gt_aligned = gt_frame - gt_root
    smpl_aligned = smpl_frame - smpl_root

    # ĐÃ SỬA: Xoay SMPL (Y-down) sang khớp với NPY (Z-up) để đo khoảng cách
    # SMPL x = NPY x | SMPL z = NPY y | -SMPL y = NPY z
    smpl_rotated = np.stack([smpl_aligned[:, 0], smpl_aligned[:, 2], -smpl_aligned[:, 1]], axis=1)

    joint_errors = {}
    total_error = 0
    count = 0

    # Chỉ tính khoảng cách Euclidean trên các khớp chung
    for h36m_idx, smpl_idx in H36M_TO_SMPL_MAP.items():
        p_gt = gt_aligned[h36m_idx]
        p_smpl = smpl_aligned[smpl_idx]

        # Khoảng cách Euclidean, tính ra đơn vị mm
        dist = np.linalg.norm(p_gt - p_smpl)*1000
        joint_errors[h36m_idx] = dist
        total_error += dist
        count += 1

    avg_error_mm = (total_error / count)
    return avg_error_mm, joint_errors

hàm Procrustes Alignment:

In [71]:
def compute_similarity_transform(S1, S2):
    """
    Tính toán phép biến đổi Procrustes để khớp S1 vào S2.
    S1, S2: mảng (N, 3) tọa độ các khớp.
    """
    # 1. Dời về tâm (Centroid alignment)
    mu1 = S1.mean(axis=0)
    mu2 = S2.mean(axis=0)
    X1 = S1 - mu1
    X2 = S2 - mu2

    # 2. Tính scale (nếu cần)
    var1 = np.sum(X1**2)
    s = np.sqrt(np.sum(X2**2) / var1)

    # 3. Tìm ma trận xoay tối ưu bằng SVD
    K = X1.T.dot(X2)
    U, w, Vt = np.linalg.svd(K)
    V = Vt.T
    R = V.dot(U.T)

    # Xử lý trường hợp bị phản chiếu (reflection)
    if np.linalg.det(R) < 0:
        V[:, 2] *= -1
        R = V.dot(U.T)

    # 4. Áp dụng biến đổi
    S1_transformed = s * X1.dot(R) + mu2
    return S1_transformed

Tính PA-MPJPE

In [72]:
def compute_pa_mpjpe(gt_frame, smpl_frame):
    """
    Tính PA-MPJPE cho 1 khung hình sau khi đã căn chỉnh Procrustes.
    """
    # Lọc các khớp tương đồng dựa trên H36M_TO_SMPL_MAP
    gt_joints = []
    smpl_joints = []
    for h36m_idx, smpl_idx in H36M_TO_SMPL_MAP.items():
        gt_joints.append(gt_frame[h36m_idx])
        smpl_joints.append(smpl_frame[smpl_idx])

    gt_joints = np.array(gt_joints)
    smpl_joints = np.array(smpl_joints)

    # Thực hiện Procrustes Alignment
    smpl_aligned = compute_similarity_transform(smpl_joints, gt_joints)

    # Tính sai số MPJPE trên các khớp đã căn chỉnh
    error = np.linalg.norm(gt_joints - smpl_aligned, axis=1)
    return np.mean(error) * 1000  # Chuyển sang mm

Tính toán lại MPJPE và PA-MPJPE sau khi đã bù trừ delta T

In [89]:
def compute_metrics_with_offset(gt_seq, pred_seq, delta_t):
    """
    Tính toán lại MPJPE và PA-MPJPE sau khi đã bù trừ Delta T
    """
    n_pred = len(pred_seq)
    n_gt = len(gt_seq)

    mpjpe_list = []
    pa_mpjpe_list = []

    for i in range(n_pred):
        gt_idx = i + delta_t

        # Chỉ tính nếu index nằm trong phạm vi của bản GT
        if 0 <= gt_idx < n_gt:
            # Lấy frame tương ứng
            gt_frame = gt_seq[gt_idx]
            pred_frame = pred_seq[i]

            # Tính MPJPE (Root-aligned)
            # Hứng 2 giá trị trả về, chỉ lấy biến err (số float)
            err, err_dict = calculate_root_aligned_mpjpe(gt_frame, pred_frame)
            mpjpe_list.append(err)

            # Tính PA-MPJPE (Procrustes-aligned)
            pa_err = compute_pa_mpjpe(gt_frame, pred_frame)
            pa_mpjpe_list.append(pa_err)

    return np.mean(mpjpe_list), np.mean(pa_mpjpe_list)

Hàm đọc tệp pkl:

In [74]:
# --- CÁC HÀM XỬ LÝ DỮ LIỆU ---
def load_pkl_data(file_path):
    """Đọc tệp PKL và trả về dữ liệu người đầu tiên."""
    data = joblib.load(file_path)
    #pdb.set_trace()
    if isinstance(data, dict):
        if 0 in data: person_data = data[0]
        elif "0" in data: person_data = data["0"]
        else: person_data = next((v for v in data.values() if isinstance(v, dict)), None)
    elif isinstance(data, list):
        person_data = next((v for v in data if isinstance(v, dict)), None)
    else:
        person_data = None

    if person_data is None:
        raise ValueError(f"Không thể đọc payload từ {file_path}")
    return person_data

Hàm downstream pkl gốc xuống để fps giảm đi một nửa:

In [75]:
def downsample_fps(person_data, original_fps=30, target_fps=15):
    """Giảm số lượng frame xuống tương ứng với target_fps."""
    num_frames = len(person_data['pose'])
    # Tính toán các index sẽ giữ lại
    indices = np.linspace(0, num_frames - 1, num=int(num_frames * (target_fps / original_fps)), dtype=int)

    new_data = copy.deepcopy(person_data)
    new_data['pose'] = person_data['pose'][indices]
    new_data['trans'] = person_data['trans'][indices]

    if len(person_data['betas']) == num_frames:
        new_data['betas'] = person_data['betas'][indices]

    return new_data

Hàm nội suy để khôi phục fps từ tệp pkl đã downstream:

In [76]:
def upsample_to_original(downsampled_data, original_num_frames):
    """Nội suy (Interpolate) dữ liệu từ 15fps lên lại số frame ban đầu."""
    curr_frames = len(downsampled_data['pose'])
    x_old = np.linspace(0, 1, curr_frames)
    x_new = np.linspace(0, 1, original_num_frames)

    new_data = copy.deepcopy(downsampled_data)

    # 1. Nội suy Dịch chuyển (trans) và Betas bằng Cubic Spline như bình thường
    cs_trans = CubicSpline(x_old, downsampled_data['trans'])
    new_data['trans'] = cs_trans(x_new)

    if len(downsampled_data['betas']) == curr_frames:
        cs_betas = CubicSpline(x_old, downsampled_data['betas'])
        new_data['betas'] = cs_betas(x_new)

    # 2. Nội suy Góc xoay (pose) bằng thuật toán SLERP
    pose_15 = downsampled_data['pose']
    # Chia (N, 72) thành (N, 24 khớp, 3 trục xoay)
    num_joints = pose_15.shape[1] // 3
    pose_reshaped = pose_15.reshape(curr_frames, num_joints, 3)
    smooth_poses = np.zeros((original_num_frames, num_joints, 3))

    for j in range(num_joints):
        # Chuyển Axis-Angle thành Quaternion
        rotations = R.from_rotvec(pose_reshaped[:, j, :])
        # Thiết lập nội suy hình cầu
        slerp = Slerp(x_old, rotations)
        # Thực hiện nội suy theo trục thời gian mới
        interp_rots = slerp(x_new)
        # Đưa về lại định dạng Axis-Angle ban đầu
        smooth_poses[:, j, :] = interp_rots.as_rotvec()

    # Làm phẳng lại thành (N, 72)
    new_data['pose'] = smooth_poses.reshape(original_num_frames, -1)
    return new_data

Hàm đọc SMPL để xác định tọa độ poses:

In [77]:
def get_all_3d_joints(model, person_data):
    """Chạy toàn bộ frame qua SMPL để lấy tọa độ 3D cùng lúc cho nhanh."""
    pose = torch.tensor(np.ascontiguousarray(person_data['pose']), dtype=torch.float32)
    trans = torch.tensor(np.ascontiguousarray(person_data['trans']), dtype=torch.float32)

    betas = person_data['betas']
    if len(betas) == 1 or len(betas) != len(pose):
        betas = np.repeat(betas[:1], len(pose), axis=0)
    betas = torch.tensor(np.ascontiguousarray(betas), dtype=torch.float32)

    global_orient = pose[:, :3]
    body_pose = pose[:, 3:]

    with torch.no_grad():
        output = model(
            betas=betas,
            global_orient=global_orient,
            body_pose=body_pose,
            transl=trans
        )
    # Lấy tọa độ các khớp [Num_frames, Num_joints, 3]
    #return output.joints.cpu().numpy()
    # Trả về cả joints (khớp) và vertices (lưới bề mặt) dưới dạng Dictionary
    return {
        'joints': output.joints.cpu().numpy(),
        'verts': output.vertices.cpu().numpy()
    }

các hàm cố gắng vẽ lại các điểm poses theo cùng 1 góc nhìn toàn cục:

In [91]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Thiết lập góc nhìn cố định toàn cục
GLOBAL_ELEV = 20
GLOBAL_AZIM = -45 # Bạn có thể chỉnh lại góc này cho giống video GT nhất

# ==========================================
# 2. HÀM PHỤ: VẼ 1 KHUNG XƯƠNG ĐỒNG NHẤT
# ==========================================
def plot_skeleton_unified(ax, pts, skeleton_type="SMPL"):
    """
    Hàm này chỉ lo việc vẽ 1 bộ xương lên 1 trục (ax) được giao.
    Xử lý luôn việc đảo trục [x, z, -y] và zoom camera.
    """
    ax.view_init(elev=GLOBAL_ELEV, azim=GLOBAL_AZIM)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])

    if skeleton_type == "H36M":
        # --- VẼ NPY (H36M) ---
        # ĐÃ SỬA: Dữ liệu NPY đã là Z-up, dùng trực tiếp [x, y, z] không đảo trục
        xs, ys, zs = pts[:, 0], pts[:, 1], pts[:, 2]

        def draw_group_h36m(bone_list, color):
            for start, end in bone_list:
                ax.plot([xs[start], xs[end]], [ys[start], ys[end]], [zs[start], zs[end]], color=color, linewidth=2)
                ax.scatter([xs[start], xs[end]], [ys[start], ys[end]], [zs[start], zs[end]], color=color, s=15)

        draw_group_h36m(BONES_H36M["trunk"], 'gray')
        draw_group_h36m(BONES_H36M["right"], 'red')
        draw_group_h36m(BONES_H36M["left"], 'blue')
        # Bounding box tự động cho NPY
        mid = np.mean(np.stack((xs, ys, zs), axis=1), axis=0)
        r = max(np.ptp(xs), np.ptp(ys), np.ptp(zs)) / 2 * 1.2
        ax.set_xlim(mid[0]-r, mid[0]+r); ax.set_ylim(mid[1]-r, mid[1]+r); ax.set_zlim(mid[2]-r, mid[2]+r)

    else:
        # --- VẼ SMPL ---
        # Lấy tọa độ bằng Dictionary map
        # Với SMPL thì VẪN PHẢI đảo trục [x, z, -y] để nó đứng lên
        joint_dict = {name: pts[idx] for name, idx in SMPL_JOINT_MAP.items() if idx < len(pts)}

        # Hàm vẽ xương theo nhóm
        def draw_group_smpl(bone_list, color):
            for b1, b2 in bone_list:
                if b1 in joint_dict and b2 in joint_dict:
                    j1, j2 = joint_dict[b1], joint_dict[b2]
                    # Đảo trục trực tiếp khi vẽ
                    ax.plot([j1[0], j2[0]], [j1[2], j2[2]], [-j1[1], -j2[1]], color=color, linewidth=2)
                    ax.scatter(j1[0], j1[2], -j1[1], color=color, s=15)
                    ax.scatter(j2[0], j2[2], -j2[1], color=color, s=15)

        draw_group_smpl(BONES_SMPL["trunk"], color='gray')
        draw_group_smpl(BONES_SMPL["right"], color='red')   # Phải = Đỏ
        draw_group_smpl(BONES_SMPL["left"], color='blue')   # Trái = Xanh

        # Bounding box bám theo Pelvis cho SMPL
        if "pelvis" in joint_dict:
            root = joint_dict["pelvis"]
            r = 1.2
            ax.set_xlim(root[0]-r, root[0]+r); ax.set_ylim(root[2]-r, root[2]+r); ax.set_zlim(-root[1]-r, -root[1]+r)

def draw_4_panels_animation(joints_npy, joints_orig, joints_down, joints_restored, output_filename, start_f, end_f):
    """
    Vẽ 4 khung hình 3D: Raw NPY (H36M) | Gốc 3D (SMPL) | Downsampled | Restored.
    """
    #num_frames = min(len(joints_npy), len(joints_orig), len(joints_restored))
    #num_frames_down = len(joints_down)
    num_frames = min(len(joints_npy), len(joints_orig['joints']), len(joints_restored['joints']))
    num_frames_down = len(joints_down['joints'])

    fig = plt.figure(figsize=(24, 6)) # 4 khung
    ax1 = fig.add_subplot(141, projection='3d') # Khung 1: Raw NPY
    ax2 = fig.add_subplot(142, projection='3d') # Khung 2: Gốc 3D SMPL
    ax3 = fig.add_subplot(143, projection='3d') # Khung 3: Downsampled SMPL
    ax4 = fig.add_subplot(144, projection='3d') # Khung 4: Restored SMPL

    axes = [ax1, ax2, ax3, ax4]
    # Mảng để lưu MPJPE nhằm tính trung bình toàn video (Mean MPJPE)
    mpjpe_orig_list, mpjpe_down_list, mpjpe_restored_list = [], [], []

    # Dictionary lưu sai số TỪNG KHỚP (khởi tạo mảng rỗng cho mỗi khớp)
    per_joint_orig = {k: [] for k in H36M_TO_SMPL_MAP.keys()}
    per_joint_down = {k: [] for k in H36M_TO_SMPL_MAP.keys()}
    per_joint_restored = {k: [] for k in H36M_TO_SMPL_MAP.keys()}

    # Bước 1: Tìm độ lệch thời gian tối ưu
    delta_t = find_optimal_time_offset(joints_npy, joints_restored['joints'])
    # Bước 2: Tính toán sai số cuối cùng sau khi đã "khớp" thời gian
    final_mpjpe, final_pa_mpjpe = compute_metrics_with_offset(joints_npy, joints_restored['joints'], delta_t)

    def update(frame_idx):
        gt_h36m_joints = joints_npy[frame_idx]
        for ax in axes: ax.clear()

        idx_down = min(int(frame_idx * (num_frames_down / num_frames)), num_frames_down - 1)

        # Trích xuất dữ liệu của frame hiện tại
        gt_pts = joints_npy[frame_idx]
        orig_pts = joints_orig['joints'][frame_idx]
        down_pts = joints_down['joints'][idx_down]
        restored_pts = joints_restored['joints'][frame_idx]

        # Tính toán MPJPE cho frame này
        err_orig, dict_orig = calculate_root_aligned_mpjpe(gt_pts, orig_pts)
        pa_err_orig = compute_pa_mpjpe(gt_pts, orig_pts)
        err_down, dict_down = calculate_root_aligned_mpjpe(gt_pts, down_pts)
        pa_err_down = compute_pa_mpjpe(gt_pts, down_pts)
        err_restored, dict_restored = calculate_root_aligned_mpjpe(gt_pts, restored_pts)
        pa_err_restored = compute_pa_mpjpe(gt_pts, restored_pts)

        # Lưu lại để tính trung bình cuối cùng
        mpjpe_orig_list.append(err_orig)
        mpjpe_down_list.append(err_down)
        mpjpe_restored_list.append(err_restored)

        # Phân bổ sai số từng khớp vào mảng lưu trữ
        for k in H36M_TO_SMPL_MAP.keys():
            per_joint_orig[k].append(dict_orig[k])
            per_joint_down[k].append(dict_down[k])
            per_joint_restored[k].append(dict_restored[k])

        # ==========================================
        # 1. VẼ KHUNG 1: DỮ LIỆU THÔ TỪ TỆP NPY (H36M)
        # ==========================================
        ax1.set_title(f"Raw NPY (Frames {start_f}-{end_f}).\nMPJPE: {final_mpjpe:.1f}, PA: {final_pa_mpjpe:.1f}")
        #ax1.view_init(elev=20., azim=-90)
        # 1. Gọi hàm phụ vẽ khung NPY
        plot_skeleton_unified(ax1, joints_npy[frame_idx], skeleton_type="H36M")

        # 2. Vẽ SMPL kèm MPJPE trên tiêu đề
        smpl_frames = [orig_pts, down_pts, restored_pts]
        titles = [
            f"Original\nMPJPE: {err_orig:.1f} | PA: {pa_err_orig:.1f}",
            f"Downsampled ({TARGET_FPS}FPS)\nMPJPE: {err_down:.1f} | PA: {pa_err_down:.1f}",
            f"Restored\nMPJPE: {err_restored:.1f}| PA: {pa_err_restored:.1f}"
        ]

        for ax, pts, title in zip(axes[1:], smpl_frames, titles[0:]):
            plot_skeleton_unified(ax, pts, skeleton_type="SMPL")
            ax.set_title(title)

        fig.suptitle(f"Frame Sync: {frame_idx + 1}/{num_frames} | Xanh=Trái, Đỏ=Phải", fontsize=15, fontweight='bold')

    # Khởi tạo Writer
    writer = animation.FFMpegWriter(
        fps=ORIGINAL_FPS,
        extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p', '-g', '1']
    )

    print(f"Đang render {num_frames} frames animation...")
    with writer.saving(fig, output_filename, dpi=100):
        for i in range(num_frames):
            update(i)
            writer.grab_frame()
            if (i + 1) % 10 == 0 or (i + 1) == num_frames:
                print(f" -> Đã render {i + 1} / {num_frames} frames...")

    plt.close(fig)
    print(f"✅ Đã lưu animation thành công tại: {output_filename}")

Hàm tạo video từ 3 animations:

In [ ]:
# --- CHƯƠNG TRÌNH CHÍNH ---
def process_pkl(input_pkl_file):
    # File đầu vào (Giả định bạn đã tải về thư mục hiện tại)
    # Thay bằng file pkl của bạn. Ở đây lấy file đầu tiên trong danh sách của bạn làm mẫu.
    # = "Axel_1_cam_3.pkl"
    ORIGINAL_FPS = auto_detect_fps(input_pkl_file)
    TARGET_FPS = ORIGINAL_FPS // 2
    original_pkl = input_pkl_file.replace(".pkl", "")
    smpl_model_path = "SMPL_NEUTRAL.pkl"
    base_name = input_pkl_file.replace(".pkl", "")
    mp4_path = f"{base_name}.mp4"
    json_path = f"{base_name}.json"
    npy_path = f"{base_name}_h36m.npy"

    # Kiểm tra sự tồn tại của bộ 3 file
    if not all(os.path.exists(p) for p in [file_name, mp4_path, json_path]):
        print(f"⚠️ Thiếu file cho {base_name}. Cần đủ .pkl, .mp4, .json")
        return

    if not os.path.exists(input_pkl_file) or not os.path.exists(smpl_model_path):
        print("Vui lòng đảm bảo các file .pkl đã được tải xuống thư mục hiện tại.")
        return

    # Đọc JSON
    with open(json_path, 'r') as f:
        meta = json.load(f)
    start_f = meta.get("source_start_frame", 0)
    end_f = meta.get("source_end_frame", 0)

    if not DETAIL_PRINT:
      print(f"🎬 Processing {base_name}: Frames {start_f} to {end_f}")

    # 2. Xử lý dữ liệu NPY (Khung 1)
    raw_npy = np.load(npy_path) # Shape (Total_Frames, 17, 3)
    joints_npy_slice = raw_npy[start_f:end_f+1]
    num_frames = len(joints_npy_slice)

    # Đọc Video Frames
    #video_frames = load_video_frames(mp4_path, start_f, end_f)

    try:
        # Lấy thời gian hiện tại với múi giờ Hồ Chí Minh
        now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
        # Định dạng thành xâu YYMMDD_hhmm
        AT_TIME = now.strftime("%y%m%d_%H%M")
        print(f"{AT_TIME}")
        if not DETAIL_PRINT:
          print("1. Đang khởi tạo mô hình SMPL...")
        model = smplx.create(smpl_model_path, model_type='smpl')
        if not DETAIL_PRINT:
          print(f"2. Đang tải dữ liệu gốc từ {input_pkl_file}...")
        orig_data = load_pkl_data(input_pkl_file)
        orig_frames = len(orig_data['pose'])
        # Đảm bảo độ dài pkl khớp với npy slice (đề phòng sai số batch)
        for k in ['pose', 'trans']:
            orig_data[k] = orig_data[k][:num_frames]
        print(f"   -> Số frame gốc: {orig_frames}")

        if not DETAIL_PRINT:
          print(f"3. Đang tạo tệp giảm FPS xuống {TARGET_FPS}...")
        data_down_fps = downsample_fps(orig_data, original_fps=ORIGINAL_FPS, target_fps=TARGET_FPS)
        file_down_fps = f"downsampled_fps_{input_pkl_file}"
        joblib.dump({0: data_down_fps}, file_down_fps)
        if not DETAIL_PRINT:
          print(f"   -> Đã lưu tệp {TARGET_FPS} FPS ({len(data_down_fps['pose'])} frames) tại: {file_down_fps}")

        if not DETAIL_PRINT:
          print(f"4. Đang nâng cấp (Upsample) tệp {TARGET_FPS} FPS trở lại số frame ban đầu...")
        restored_data = upsample_to_original(data_down_fps, original_num_frames=orig_frames)
        file_restored = f"restored_{ORIGINAL_FPS}fps_{input_pkl_file}"
        joblib.dump({0: restored_data}, file_restored)
        if not DETAIL_PRINT:
          print(f"   -> Đã lưu tệp phục hồi ({len(restored_data['pose'])} frames) tại: {file_restored}")

        if not DETAIL_PRINT:
          print("5. Đang trích xuất tọa độ 3D để tạo Video Animation...")
        joints_orig = get_all_3d_joints(model, orig_data)
        joints_down = get_all_3d_joints(model, data_down_fps) # Lấy thêm toạ độ 3D của bản downstream fps
        joints_restored = get_all_3d_joints(model, restored_data)

        if not DETAIL_PRINT:
          print("6. Đang tạo Video ghép 3 animation cạnh nhau...")
        # Đã cập nhật thành hàm 3 khung hình
        #create_three_panel_animation(joints_orig, joints_down, joints_restored, output_filename="comp_" + original_pkl + "_" + AT_TIME + ".mp4")
        # Xuất animation 4 khung hình
        # 5. Gọi hàm vẽ tách biệt
        out_name = f"comp_{base_name}_{AT_TIME}.mp4"
        #create_four_panel_3d_animation(
        draw_4_panels_animation(
            joints_npy_slice,
            joints_orig,
            joints_down,
            joints_restored,
            out_name,
            start_f, end_f
        )
        if not DETAIL_PRINT:
          print("\nHOÀN TẤT TẤT CẢ CÁC BƯỚC!")

    except Exception as e:
        if not DETAIL_PRINT:
          print(f"\n[Lỗi trong quá trình xử lý]: {e}")

if __name__ == "__main__":
    # Duyệt qua tất cả các tên tệp (keys) trong dictionary
    for file_name in GOOGLE_DRIVE_IDS.keys():

        # Lọc: Chỉ lấy file có đuôi .pkl VÀ không phải là file model SMPL_NEUTRAL
        if file_name.endswith('.pkl') and file_name != "SMPL_NEUTRAL.pkl":

            print(f"\n{'='*60}")
            if not DETAIL_PRINT:
              print(f"🚀 BẮT ĐẦU XỬ LÝ TỆP: {file_name}")
            print(f"{'='*60}")

            # Gọi hàm xử lý và truyền tên tệp vào
            process_pkl(file_name)
            #pdb.set_trace()

    print("\n🎉 ĐÃ XỬ LÝ XONG TOÀN BỘ DANH SÁCH TỆP!")